# BDD-A as a **negative-bag pool** for T2 (DADA-2000 original) — feasibility EDA

Plan: `.project/plans/katvad-bdda-normal-bag-eda.md` · Template: `colab/D2City/eda_normal_bags.ipynb`
(read-out `core/docs/D2CITY_EDA.md`, lesson **C38**).

**Question.** Can BDD-A dashcam clips be **added to T2 as negative bags**? Do BDD-A negatives teach a
linear reader of the frozen CLIP features the **accident**, or the **source**? D2City (same country
as DADA) failed at Δ −0.090 with shortcut AUC 1.000, so **NO-GO is the prior**. This run is cheap
(3.1 GB, one feature pass, no training), and it is how that prior gets tested.

**BDD-A ≠ BDD100K.** BDD-A (Berkeley DeepDrive *Attention*) was collected around **braking events**.
The GPS splits it into two arms: `all`, and `calm` (no hard braking). **The gate is read on `calm`.**

**GPU runtime** (T4 is enough). Nothing in `core/` changes; no model is trained.

| § | gate | bar (pre-registered, plan §4) |
|---|---|---|
| 1 | **G-I** inventory (HARD) | decodable ≥ 99 % · step-matched ≥ 99 % · duration ≥ 5.33 s on ≥ 99 % · GPS-known ≥ 80 % · calm ≥ 300 clips |
| 3 | **G-L** length leak (HARD, L-T2) | clip-level length AUC ∈ [0.45, 0.55] |
| 4 | sanity | R0 `auc_macro` ∈ [0.62, 0.68] (soft: = 0.676258, the D2City run's R0) |
| 4 | **G-X** (on `calm`) | PASS: X ≥ 0.60 and Δ(X−R0) ≥ −0.03 · FAIL: X < 0.55 or Δ < −0.06 |
| 4 | **G-M** (on `calm`) | Δ(M−R0) ≥ −0.01 |
| 4 | S, S-ref, shortcut | descriptive (shortcut red flag > 0.90) |

**Drive layout expected:** `Thesis/data/BDDA/archive.zip`, whose members are `training/{camera_videos,gps_jsons}/…`
(macOS `__MACOSX/` and `._*` entries are skipped). Any other `*.zip` in that folder is unpacked too, so a
validation zip with members `validation/…` is picked up automatically. Unzipped `training/` / `validation/` folders
on Drive still work.
Bring back: `outputs/EDA/BDDA/{eda_bdda.json, eda_bdda.md, probes.json, inventory.csv, montage.png, autocorr_seconds.png}`.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os
import sys
from pathlib import Path

DRIVE = '/content/drive/MyDrive/Thesis'
os.environ['PROJECT_ROOT']       = DRIVE
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE}/outputs'
for v in ('KATVAD_DATA_ROOT', 'KATVAD_CACHE_ROOT', 'KATVAD_OUTPUT_ROOT'):
    os.makedirs(os.environ[v], exist_ok=True)

os.environ['REPO'] = f'{DRIVE}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core' / 'eda' / 'protocol.py').is_file(), f'no core/ checkout at {REPO} -- sync it first'
os.environ['PYTHONPATH'] = str(REPO)
sys.path.insert(0, str(REPO))

# core.constants reads KATVAD_* at import time -> import after the block above.
from core import constants  # noqa: E402

# --- BDD-A (the candidate negative pool) -----------------------------------------------
BDDA_ROOT = constants.DATA_ROOT / 'BDDA'
BDDA_SPLITS = ('training', 'validation')       # validation is optional; ids get a split prefix (C26)
# One cache dir per transform + time step (C2): _ncc, 0.267 s/step (stride 8 @ 30 fps, 16 @ 60 fps).
BDDA_CLIP = constants.CLIP_CACHE_DIR / 'BDDA_dt267_ncc'

# --- DADA-2000 original (positives) -- READ ONLY --------------------------------------
DADA_DATASET = constants.DADA_ORIGIN_DATASET                  # 'DADA2000_orig'
DADA_CLIP = constants.CLIP_CACHE_DIR / DADA_DATASET           # keyed by SOURCE clip, stride 8, ncc
DADA_T2 = constants.DATA_ROOT / DADA_DATASET                  # T2 corpus: meta.json = one row per window
DADA_FRAMES = constants.DATA_ROOT / 'DADA2000Origin' / constants.DADA_ORIGIN_ROOT_DIRNAME  # montage only

# --- DoTA (S-ref only; never trained on) -----------------------------------------------
DOTA_DATA = constants.DATA_ROOT / 'DoTA' / 'labels_s8'
DOTA_CLIP = constants.CLIP_CACHE_DIR / 'DoTA_s8_ncc'

OUT = constants.OUTPUT_ROOT / 'EDA' / 'BDDA'                  # Drive: survives the runtime
WORK = Path('/content/bdda')                                  # VM-local disk, never Drive
OUT.mkdir(parents=True, exist_ok=True)
WORK.mkdir(parents=True, exist_ok=True)

BDDA_ZIPS = sorted(BDDA_ROOT.glob('*.zip'))                   # archive.zip = training/{camera_videos,gps_jsons}
print(f'  BDDA zips         {[z.name for z in BDDA_ZIPS] or "none -- falling back to unzipped folders"}')
for name, path in (('BDDA training', BDDA_ROOT / 'training'), ('BDDA validation*', BDDA_ROOT / 'validation'),
                   ('DADA clip', DADA_CLIP), ('DADA T2 corpus', DADA_T2), ('DADA frames*', DADA_FRAMES),
                   ('DoTA data', DOTA_DATA), ('DoTA clip', DOTA_CLIP), ('out', OUT)):
    print(f'  {name:17s} {path}   {"OK" if path.exists() else "MISSING"}')
print('  (* optional: validation adds clips; DADA frames only feed the montage.'
      ' The BDDA folders may be MISSING when the zip is used.)')

In [ ]:
%%bash
pip install -q "transformers==4.56.*" av
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU -- section 2 will be very slow'
df -h /content | tail -1

### 0.1 Pre-registered constants (plan §4) — edit nothing here after the first run

The time step is **derived, not chosen**: DADA samples every 8 frames at 30 fps (assumption **A1**,
literature only) = 0.267 s. BDD-A mixes ~30 fps and ~60 fps clips, so its stride is **per clip**:
`round(fps × 0.267)` → 8 or 16. A clip whose step misses by more than 5 % is dropped, with a count.

In [ ]:
import json
import logging

import numpy as np

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(name)s: %(message)s')
LOG = logging.getLogger('bdda_eda')

SEED = constants.SEED
DADA_FPS = 30.0                                  # A1 (literature, unverified)
DADA_STRIDE = constants.FRAME_STRIDE             # 8 -- the DADA cache's stride, fixed (C2)
DADA_STEP_S = DADA_STRIDE / DADA_FPS             # 0.267 s
STEP_TOL = 0.05                                  # |stride/fps - DADA_STEP_S| / DADA_STEP_S
T2_W = constants.DADA_ORIGIN_WINDOW_LENGTH       # 20 sampled frames
T2_HOP = constants.DADA_ORIGIN_WINDOW_STRIDE     # 8


def clip_stride(fps: float) -> int:
    return max(1, round(fps * DADA_STEP_S))


# G-I
GI_DECODABLE = 0.99
GI_STEP_SHARE = 0.99
GI_MIN_DURATION_S = T2_W * DADA_STEP_S           # one T2 window = 5.33 s
GI_DURATION_SHARE = 0.99
GI_GPS_KNOWN = 0.80
GI_CALM_MIN_CLIPS = 300                          # below -> add data/BDDA/validation
# GPS arm (plan D3)
HARD_BRAKE_MPS2 = -3.0                           # ~0.3 g; a clip is `calm` iff its min accel >= this
GPS_DT_S = (0.5, 1.5)                            # consecutive 1 Hz pairs only; longer gaps are not an accel
# G-L
GL_BAND = (0.45, 0.55)
TEST_MIXES = (0.0, 0.25, 0.5, 1.0)               # BDD-A windows per T2 test window
T2_ORACLE_REF = 0.7037                           # T2 test clip oracle (Gate W) -- soft check at mix 0
# Probes
FOLDS = constants.EDA_PROBE_FOLDS                # 5
T95 = 2.776                                      # two-sided t, FOLDS - 1 = 4 df
R0_SANITY = (0.62, 0.68)
R0_D2CITY = 0.676258                             # R0 never reads the pool -> must reproduce (plan D6)
R0_REPRO_TOL = 1e-4
GX_PASS, GX_FAIL = 0.60, 0.55
GX_DELTA_PASS, GX_DELTA_MARGINAL = -0.03, -0.06
GM_DELTA = -0.01
SHORTCUT_RED = 0.90
GATE_ARM = 'calm'
ARMS = ('all', 'calm')
# Mechanics (no effect on any number)
INVENTORY_WORKERS = 8
DECODE_WORKERS = 3
ENCODE_BATCH = 64
MONTAGE_N = 8

for fps in (29.97, 30.0, 59.94, 60.0):
    s = clip_stride(fps)
    print(f'  {fps:5.2f} fps -> stride {s:2d} = {s / fps:.4f} s ({s / fps / DADA_STEP_S - 1:+.2%} vs DADA)')

REPORT: dict = {'plan': '.project/plans/katvad-bdda-normal-bag-eda.md',
                'config': {'seed': SEED, 'dada_fps_assumed': DADA_FPS, 'dada_stride': DADA_STRIDE,
                           'step_s': DADA_STEP_S, 'step_tol': STEP_TOL, 'hard_brake_mps2': HARD_BRAKE_MPS2,
                           't2_window': T2_W, 't2_hop': T2_HOP, 'folds': FOLDS, 'gate_arm': GATE_ARM,
                           'probe_C': constants.EDA_PROBE_C,
                           'probe_max_iter': constants.EDA_PROBE_MAX_ITER}}


def save_report() -> None:
    # Written after every section, so a dead runtime keeps what was measured.
    (OUT / 'eda_bdda.json').write_text(json.dumps(REPORT, indent=2, default=float))

## 1. Inventory — Gate **G-I** (HARD) + the GPS arms

Unzip (or copy) to VM-local disk, then count **files**, not folders (C10). Every clip is opened and its first
frame decoded. The GPS gives each clip its minimum acceleration over consecutive ~1 s pairs. A clip
is `calm` iff that minimum is ≥ −3 m/s². A clip with no usable GPS is kept in `all` and left out of `calm`.

In [ ]:
import shutil
import zipfile


def is_junk(member: str) -> bool:
    # macOS zips carry AppleDouble twins (`__MACOSX/.../._10.mp4`): 4 KB, not videos, not JSON.
    return member.startswith('__MACOSX/') or '/._' in f'/{member}' or member.endswith('.DS_Store')


for zp in BDDA_ZIPS:
    marker = WORK / f'.{zp.name}.done'
    if marker.exists():
        continue
    with zipfile.ZipFile(zp) as zf:
        members = [m for m in zf.namelist() if not is_junk(m)]
        zf.extractall(WORK, members=members)       # Drive -> VM-local disk, never decode from Drive
    marker.touch()
    print(f'  unzipped {zp.name}: {len(members)} members ({len(zf.namelist()) - len(members)} macOS junk skipped)')

VIDEO_PATHS, GPS_PATHS = {}, {}
for split in BDDA_SPLITS:
    dst = WORK / split
    src = BDDA_ROOT / split
    if not (dst / 'camera_videos').is_dir() and (src / 'camera_videos').is_dir():
        for sub in ('camera_videos', 'gps_jsons'):            # fallback: unzipped folders on Drive
            if (src / sub).is_dir():
                shutil.copytree(src / sub, dst / sub, dirs_exist_ok=True)
    if not (dst / 'camera_videos').is_dir():
        print(f'  {split}: absent -- skipped')
        continue
    for p in sorted((dst / 'camera_videos').glob('*.mp4')):
        if p.name.startswith('._'):
            continue
        vid = f'{split}_{p.stem}'                 # BDD-A ids are bare integers per split (C26)
        VIDEO_PATHS[vid] = p
        GPS_PATHS[vid] = dst / 'gps_jsons' / f'{p.stem}.json'
    print(f'  {split}: {sum(v.startswith(split + "_") for v in VIDEO_PATHS)} mp4, '
          f'{len(list((dst / "gps_jsons").glob("[!._]*.json")))} gps json')
assert VIDEO_PATHS, f'no mp4 found -- expected {BDDA_ROOT}/archive.zip with training/camera_videos/*.mp4'
print(f'{len(VIDEO_PATHS)} clips total')

In [ ]:
import csv
from concurrent.futures import ThreadPoolExecutor

import av


def probe(item: tuple[str, Path]) -> dict:
    vid, path = item
    row = {'id': vid, 'ok': False}
    try:
        with av.open(str(path)) as container:
            stream = container.streams.video[0]
            fps = float(stream.average_rate) if stream.average_rate else float('nan')
            duration = (float(stream.duration * stream.time_base) if stream.duration
                        else float(container.duration) / av.time_base)
            frames = stream.frames or round(duration * fps)
            next(container.decode(stream))
        stride = clip_stride(fps)
        row.update(ok=True, fps=fps, width=stream.codec_context.width, height=stream.codec_context.height,
                   frames=int(frames), duration_s=duration, codec=stream.codec_context.name, stride=stride,
                   step_err=stride / fps / DADA_STEP_S - 1)
    except (av.error.FFmpegError, StopIteration, OSError, ValueError, IndexError) as exc:
        row['error'] = repr(exc)
    return row


def gps_profile(path: Path) -> dict:
    if not path.is_file():
        return {'gps': 'no_file'}
    try:
        locs = json.loads(path.read_text())['locations']
        t = np.array([p['timestamp'] for p in locs], dtype=float) / 1000.0
        s = np.array([p['speed'] for p in locs], dtype=float)
    except (json.JSONDecodeError, KeyError, TypeError, OSError) as exc:
        return {'gps': f'unreadable: {exc!r}'}
    if len(t) < 2:
        return {'gps': 'too_short', 'gps_samples': len(t)}
    dt = np.diff(t)
    keep = (dt >= GPS_DT_S[0]) & (dt <= GPS_DT_S[1])
    if not keep.any():
        return {'gps': 'no_1hz_pairs', 'gps_samples': len(t)}
    accel = np.diff(s)[keep] / dt[keep]
    return {'gps': 'ok', 'gps_samples': len(t), 'min_accel': float(accel.min()),
            'mean_speed': float(s.mean())}


with ThreadPoolExecutor(INVENTORY_WORKERS) as pool:
    INVENTORY = list(pool.map(probe, sorted(VIDEO_PATHS.items())))
for row in INVENTORY:
    row.update(gps_profile(GPS_PATHS[row['id']]))
    row['calm'] = row['gps'] == 'ok' and row['min_accel'] >= HARD_BRAKE_MPS2

with (OUT / 'inventory.csv').open('w', newline='') as fh:
    keys = ['id', 'ok', 'fps', 'stride', 'step_err', 'width', 'height', 'frames', 'duration_s', 'codec',
            'gps', 'gps_samples', 'min_accel', 'mean_speed', 'calm', 'error']
    writer = csv.DictWriter(fh, fieldnames=keys, extrasaction='ignore')
    writer.writeheader()
    writer.writerows(INVENTORY)

n = len(INVENTORY)
good = [r for r in INVENTORY if r['ok']]
step_ok = [r for r in good if abs(r['step_err']) <= STEP_TOL]
USABLE = {r['id']: r for r in step_ok if r['duration_s'] >= GI_MIN_DURATION_S}
gps_status: dict[str, int] = {}
for r in INVENTORY:
    gps_status[r['gps']] = gps_status.get(r['gps'], 0) + 1
calm_ids = sorted(v for v, r in USABLE.items() if r['calm'])
accel = np.array([r['min_accel'] for r in good if r['gps'] == 'ok'])
res: dict[str, int] = {}
fps_hist: dict[str, int] = {}
for r in good:
    res[f"{r['width']}x{r['height']}"] = res.get(f"{r['width']}x{r['height']}", 0) + 1
    fps_hist[f"{r['fps']:.2f}"] = fps_hist.get(f"{r['fps']:.2f}", 0) + 1
pct = lambda a: dict(zip(map(str, constants.EDA_PERCENTILES), np.percentile(a, constants.EDA_PERCENTILES).tolist()))

checks = {'decodable': (len(good) / n, GI_DECODABLE),
          'step_matched': (len(step_ok) / n, GI_STEP_SHARE),
          'duration_ge_one_window': (sum(r['duration_s'] >= GI_MIN_DURATION_S for r in good) / n, GI_DURATION_SHARE),
          'gps_known': (gps_status.get('ok', 0) / n, GI_GPS_KNOWN),
          'calm_clips': (len(calm_ids), GI_CALM_MIN_CLIPS)}
gi_pass = all(v >= bar for v, bar in checks.values())
REPORT['G-I'] = {'clips': n, 'usable': len(USABLE), 'calm': len(calm_ids),
                 'checks': {k: {'value': v, 'bar': bar, 'pass': v >= bar} for k, (v, bar) in checks.items()},
                 'resolution': res, 'fps': fps_hist, 'gps_status': gps_status,
                 'duration_s_pct': pct([r['duration_s'] for r in good]),
                 'min_accel_pct': pct(accel), 'mean_speed_pct': pct([r['mean_speed'] for r in good if r['gps'] == 'ok']),
                 'failed': {r['id']: r.get('error') for r in INVENTORY if not r['ok']}, 'pass': gi_pass}
save_report()
print(json.dumps({k: v for k, v in REPORT['G-I'].items() if k != 'failed'}, indent=1))
print(f"failed: {REPORT['G-I']['failed']}")
if len(calm_ids) < GI_CALM_MIN_CLIPS:
    print('!! too few calm clips -> upload data/BDDA/validation (camera_videos + gps_jsons) and re-run')
assert gi_pass, 'G-I FAILED -- read the checks above before anything below'
print('G-I PASS')

### 1.1 Montage — **look at it** (human check)

Row 1: BDD-A `calm` clips under the project transform (full frame squashed to 224², `_ncc`).
Row 2: BDD-A hard-braking clips. Row 3: DADA (only if its frames are on Drive). Look for timestamps,
logos, hood, dashboard reflections, colour cast: anything a linear probe could read instead of the road.

In [ ]:
import random

import matplotlib.pyplot as plt
from PIL import Image

from core.data.video_io import list_frame_images, read_sampled_frames


def squash(frame: np.ndarray) -> np.ndarray:
    size = constants.CROP_SIZE
    return np.asarray(Image.fromarray(frame).resize((size, size), Image.BILINEAR))


def middle_frame(vid: str) -> np.ndarray:
    frames = read_sampled_frames(VIDEO_PATHS[vid], stride=USABLE[vid]['stride'])
    return frames[len(frames) // 2]


rng = random.Random(SEED)
braking_ids = sorted(v for v, r in USABLE.items() if r['gps'] == 'ok' and not r['calm'])
rows = [[squash(middle_frame(v)) for v in rng.sample(calm_ids, min(MONTAGE_N, len(calm_ids)))],
        [squash(middle_frame(v)) for v in rng.sample(braking_ids, min(MONTAGE_N, len(braking_ids)))]]
labels = ['BDD-A calm', 'BDD-A hard braking']
if DADA_FRAMES.exists():
    folders = sorted(DADA_FRAMES.glob(f'*/*/{constants.DADA_ORIGIN_IMAGES_SUBDIR}'))
    dada_row = []
    for folder in rng.sample(folders, min(MONTAGE_N, len(folders))):
        imgs = list_frame_images(folder)
        dada_row.append(squash(np.asarray(Image.open(imgs[len(imgs) // 2]).convert('RGB'))))
    rows.append(dada_row)
    labels.append('DADA (ncc)')
fig, axes = plt.subplots(len(rows), MONTAGE_N, figsize=(2 * MONTAGE_N, 2.2 * len(rows)))
for r, (row, label) in enumerate(zip(rows, labels)):
    for c in range(MONTAGE_N):
        ax = axes[r][c]
        ax.axis('off')
        if c < len(row):
            ax.imshow(row[c])
    axes[r][0].set_title(label, loc='left', fontsize=9)
fig.tight_layout()
fig.savefig(OUT / 'montage.png', dpi=110)
plt.show()

## 2. Extraction — BDD-A CLIP features, 0.267 s/step, `_ncc`

One cache for both GPS arms: the arms only select clips. Pixels are preprocessed **in batches** (C9).
Writes are atomic and resumable (`save_array` / `is_complete`, C11) and go straight to Drive. DADA's
cache is not touched (C2).

In [ ]:
import time

import torch

from core.device import resolve_device
from core.tools.extract_clip_features import load_pretrained_encoder, preprocess_frames
from core.tools.feature_cache import is_complete, save_array

DEVICE = resolve_device('auto')
ENCODER = load_pretrained_encoder(DEVICE)
BDDA_CLIP.mkdir(parents=True, exist_ok=True)


@torch.no_grad()
def encode(frames: np.ndarray) -> np.ndarray:
    chunks = []
    for start in range(0, len(frames), ENCODE_BATCH):
        pixels = preprocess_frames(frames[start:start + ENCODE_BATCH], constants.CROP_SIZE,
                                   center_crop=False).to(DEVICE)
        chunks.append(ENCODER(pixel_values=pixels).image_embeds.float().cpu())
    return torch.cat(chunks).numpy().astype(np.float32)


ok_ids = sorted(USABLE)
pending = [v for v in ok_ids if not is_complete(BDDA_CLIP / f'{v}.npy')]
print(f'resume: {len(ok_ids) - len(pending)}/{len(ok_ids)} done, {len(pending)} to go')

failed, t0 = {}, time.time()
with ThreadPoolExecutor(DECODE_WORKERS) as pool:
    futures = {}
    queue = list(pending)
    while queue or futures:
        while queue and len(futures) < DECODE_WORKERS + 1:   # bounded RAM
            vid = queue.pop(0)
            futures[vid] = pool.submit(read_sampled_frames, VIDEO_PATHS[vid], USABLE[vid]['stride'])
        vid = next(iter(futures))
        try:
            frames = futures.pop(vid).result()
        except (av.error.FFmpegError, OSError, ValueError) as exc:
            failed[vid] = repr(exc)
            LOG.warning('decode failed %s: %s', vid, exc)
            continue
        save_array(BDDA_CLIP / f'{vid}.npy', encode(frames))
        done = len(pending) - len(queue) - len(futures)
        if done % 50 == 0:
            rate = done / (time.time() - t0)
            print(f'  {done}/{len(pending)}  {rate:.2f} clips/s  eta {(len(pending) - done) / max(rate, 1e-9) / 60:.0f} min')

manifest = {'step_s': DADA_STEP_S, 'stride_by_clip': {v: USABLE[v]['stride'] for v in ok_ids},
            'transform': 'no_center_crop', 'crop_size': constants.CROP_SIZE,
            'clip_model': constants.CLIP_MODEL_NAME, 'clip_revision': constants.CLIP_MODEL_REVISION,
            'clips_ok': len(ok_ids) - len(failed), 'decode_failed': failed}
(BDDA_CLIP / 'extraction_manifest.json').write_text(json.dumps(manifest, indent=2))
REPORT['extraction'] = {k: v for k, v in manifest.items() if k != 'stride_by_clip'}
save_report()
print(f'done; {len(failed)} late decode failures (logged, excluded)')

## 3. Load both sides, then the length / oracle **lever table** — Gate **G-L**

DADA labels are rebuilt **exactly as the T2 pipeline builds them**: `DadaRecord` + `sampled_frame_labels` on
the source clip, then each T2 window is `labels[start:end]` of its source. Each window's positive count is
checked against the `positive_frames` that T2's `meta.json` recorded. The DADA side is the same as in the
D2City run, so R0 must reproduce.

In [ ]:
import matplotlib.pyplot as plt

from core.data.dada import DadaRecord, sampled_frame_labels
from core.eda.corpus import describe, load_dataset_files
from core.tools.feature_cache import is_complete   # sections 3+ run without section 2 on a resume

t2_meta = json.loads((DADA_T2 / constants.META_FILENAME).read_text())
sources: dict[str, dict] = {}
for wid, m in t2_meta.items():
    src = m.get('source', wid)
    if m.get('normalized_span') is not None:
        sources.setdefault(src, m)

SRC_LABELS, DADA_FEATS, DADA_LABELS, mismatch, missing = {}, {}, {}, [], []
for src, m in sorted(sources.items()):
    record = DadaRecord(video_id=src, folder_name=src, class_name=m['class_name'],
                        fault_label=m['fault_label'], total_frames=int(m['total_frames']),
                        span=tuple(m['normalized_span']), accident_frac=m.get('accident_frac'))
    labels = np.asarray(sampled_frame_labels(record, DADA_STRIDE), dtype=np.int8)
    SRC_LABELS[src] = labels
    path = DADA_CLIP / f'{src}.npy'
    if not path.is_file():
        missing.append(src)
        continue
    feats = np.load(path)
    if len(feats) != len(labels):
        mismatch.append(f'{src}: {len(feats)} rows vs {len(labels)} labels')
        continue
    if labels.any():
        DADA_FEATS[src], DADA_LABELS[src] = feats, labels

print(f'DADA sources {len(sources)} -> usable {len(DADA_FEATS)} | missing features {len(missing)} '
      f'| row mismatch {len(mismatch)}')
assert len(mismatch) <= 0.01 * len(sources), f'row mismatches {mismatch[:5]} -- wrong cache for these labels (C2)'
dada_len = np.array([len(v) for v in DADA_LABELS.values()])

# T2 test windows, labels sliced from their source; checked against meta's positive_frames.
T2_TEST, t2_bad = [], []
for wid, m in sorted(t2_meta.items()):
    if m.get('split') != 'test' or m.get('source') not in SRC_LABELS:
        continue
    lab = SRC_LABELS[m['source']][m['start']:m['end']]
    if m.get('positive_frames') is not None and int(lab.sum()) != int(m['positive_frames']):
        t2_bad.append(wid)
    T2_TEST.append(lab)
print(f'T2 test windows {len(T2_TEST)} | positive-count mismatches {len(t2_bad)}')
assert len(t2_bad) <= 0.01 * max(len(T2_TEST), 1), f'T2 label rebuild disagrees with meta.json: {t2_bad[:5]}'

BDDA_FEATS = {}
for vid in sorted(USABLE):
    path = BDDA_CLIP / f'{vid}.npy'
    if is_complete(path):
        BDDA_FEATS[vid] = np.load(path)
ARM_IDS = {'all': sorted(BDDA_FEATS), 'calm': sorted(v for v in BDDA_FEATS if USABLE[v]['calm'])}
bdda_len = {v: len(a) for v, a in BDDA_FEATS.items()}
print(describe(dada_len.tolist(), 'DADA sampled length (whole source)'))
print(describe(list(bdda_len.values()), 'BDD-A sampled length (whole clip)'))
print({arm: len(ids) for arm, ids in ARM_IDS.items()})

DOTA_PRE = {}
try:
    dota_labels = load_dataset_files(DOTA_DATA, 'DoTA').frame_labels_test
except FileNotFoundError as exc:
    dota_labels = {}
    print(f'DoTA unavailable -> S-ref skipped ({exc})')
for vid, lab in dota_labels.items():
    lab = np.asarray(lab)
    path = DOTA_CLIP / f'{vid}.npy'
    if lab.any() and path.is_file():
        first = int(np.argmax(lab))
        feats = np.load(path)
        if first >= 1 and len(feats) == len(lab):
            DOTA_PRE[vid] = feats[:first]
print(f'DoTA clips with >= 1 pre-anomaly frame: {len(DOTA_PRE)}')
REPORT['data'] = {'dada_sources': len(sources), 'dada_usable': len(DADA_FEATS),
                  'dada_missing_features': len(missing), 'dada_row_mismatch': len(mismatch),
                  'dada_length': describe(dada_len.tolist(), 'DADA sampled length'),
                  't2_test_windows': len(T2_TEST), 't2_label_mismatch': len(t2_bad),
                  'bdda_clips': {arm: len(ids) for arm, ids in ARM_IDS.items()},
                  'bdda_length': describe(list(bdda_len.values()), 'BDD-A length'),
                  'dota_pre_clips': len(DOTA_PRE)}
save_report()

In [ ]:
from core.eda.protocol import clip_constant_oracle, clip_length_leak


def t2_windows(ids: list[str]) -> list[tuple[str, int, int]]:
    # T2's own geometry on a negative-only clip: W=20, hop 8, windows fully inside the clip.
    return [(v, s, s + T2_W) for v in ids for s in range(0, bdda_len[v] - T2_W + 1, T2_HOP)]


def lever_row(pos_bags: list[np.ndarray], neg_lengths: list[int], mix: float, seed: int = SEED) -> dict:
    rng = np.random.default_rng(seed)
    n_neg = min(len(neg_lengths), round(mix * len(pos_bags)))
    chosen = [neg_lengths[i] for i in rng.choice(len(neg_lengths), n_neg, replace=False)] if n_neg else []
    labels = pos_bags + [np.zeros(k, dtype=np.int8) for k in chosen]
    row = {'mix': mix, 'bdda_bags': n_neg, 'oracle_micro': clip_constant_oracle(labels)['auc_micro']}
    if n_neg:
        leak = clip_length_leak(labels)
        row.update(length_auc=leak['auc_clip_level'], length_micro=leak['auc_micro'],
                   direction=leak['direction'])
    return row


GATE_WINDOWS = t2_windows(ARM_IDS[GATE_ARM])
too_short = sum(bdda_len[v] < T2_W for v in ARM_IDS[GATE_ARM])
dada_src_bags = [DADA_LABELS[v] for v in sorted(DADA_LABELS)]
LEVER = {
    # contrast only: whole BDD-A clip vs whole DADA source (a training corpus would never do this)
    'L-raw': [lever_row(dada_src_bags, [bdda_len[v] for v in ARM_IDS[GATE_ARM]], m) for m in TEST_MIXES],
    # chosen: T2 test windows + BDD-A cut into T2's own windows
    'L-T2': [lever_row(T2_TEST, [e - s for _, s, e in GATE_WINDOWS], m) for m in TEST_MIXES],
}
for recipe, rows_ in LEVER.items():
    print(recipe)
    for row in rows_:
        print('   ', {k: round(v, 4) if isinstance(v, float) else v for k, v in row.items()})

CHOSEN = 'L-T2'
gl_auc = next(r['length_auc'] for r in LEVER[CHOSEN] if r['mix'] == 1.0)
gl_pass = GL_BAND[0] <= gl_auc <= GL_BAND[1]
t2_oracle0 = LEVER[CHOSEN][0]['oracle_micro']
print(f'T2 mix-0 oracle {t2_oracle0:.4f} (Gate W measured {T2_ORACLE_REF}) '
      f'{"ok" if abs(t2_oracle0 - T2_ORACLE_REF) < 0.005 else "!! DIFFERS -- check the T2 corpus version"}')
print(f'{GATE_ARM}: {len(GATE_WINDOWS)} BDD-A windows from {len(ARM_IDS[GATE_ARM])} clips ({too_short} clips < W, dropped)')
REPORT['G-L'] = {'lever': LEVER, 'chosen': CHOSEN, 'bdda_windows': len(GATE_WINDOWS), 'clips_shorter_than_W': too_short,
                 't2_oracle_mix0': t2_oracle0, 'length_auc_mix1': gl_auc, 'pass': gl_pass}
save_report()
print(f'G-L {CHOSEN}: length AUC {gl_auc:.4f} in {GL_BAND} -> {"PASS" if gl_pass else "FAIL"}')
assert gl_pass, 'G-L FAILED -- read the lever table; do NOT run the probes on a leaking recipe'

### 3.1 Time scale check (A1) — feature change per **second**

Both sides are sampled 0.267 s apart. If A1 holds and the content is alike, the curves overlap. A BDD-A
curve well **above** DADA's means its scenes change more slowly, which is one more source cue.

In [ ]:
from core.eda.features import temporal_autocorrelation

dada_pre = {v: DADA_FEATS[v][:int(np.argmax(DADA_LABELS[v]))] for v in DADA_FEATS}
dada_pre = {v: a for v, a in dada_pre.items() if len(a) > 1}
AUTOCORR = {'DADA pre-accident': temporal_autocorrelation(dada_pre),
            **{f'BDD-A {arm}': temporal_autocorrelation({v: BDDA_FEATS[v] for v in ARM_IDS[arm]}) for arm in ARMS}}
fig, ax = plt.subplots(figsize=(6, 4))
REPORT['autocorr'] = {}
for name, ac in AUTOCORR.items():
    lags = sorted(ac['cosine_by_lag'], key=lambda k: int(k[3:]))
    REPORT['autocorr'][name] = {'step_s': DADA_STEP_S, 'cosine_by_lag': ac['cosine_by_lag']}
    ax.plot([int(k[3:]) * DADA_STEP_S for k in lags], [ac['cosine_by_lag'][k]['mean'] for k in lags],
            marker='o', label=name)
ax.set_xlabel('seconds between frames')
ax.set_ylabel('mean cosine(f_t, f_t+lag)')
ax.legend()
fig.tight_layout()
fig.savefig(OUT / 'autocorr_seconds.png', dpi=110)
plt.show()
save_report()

## 4. Probes — **R0**, **X**, **M** per arm, plus **S**, S-ref

Every probe: standardized logistic regression, `C = EDA_PROBE_C`, balanced classes, folds grouped by
**source clip**, **the same folds for every probe**, so Δ(X−R0) and Δ(M−R0) are paired per fold
(t95 over 5 folds, 4 df). BDD-A negatives are **whole clips**, since a frame probe does not see bag length (plan D5).

- **R0**: DADA in-span = 1 vs DADA out-of-span (in-video) = 0. Fitted **once** because it never reads BDD-A.
- **X_{arm}**: DADA in-span = 1 vs **BDD-A only** = 0. **The gate, read on `calm`.**
- **M_{arm}**: DADA in-span = 1 vs DADA out-of-span **+** BDD-A = 0. This asks whether BDD-A helps when added.
- Scored on held-out DADA videos: `auc_macro` (within-video), plus **shortcut AUC** =
  P(score(DADA normal frame) > score(BDD-A frame)) on held-out folds.
- **S_{arm}**: DADA pre-accident vs BDD-A (pooled AUC). **S-ref**: DADA pre-accident vs DoTA pre-anomaly.

In [ ]:
import ctypes
import gc
import time

import psutil
import torch
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

from core.metrics import frame_auc

# Section 2 leaves the CLIP encoder, a CUDA context and PyAV decode buffers in this process (pending (w)).
for _name in ('ENCODER', 'frames', 'futures', 'rows'):
    globals().pop(_name, None)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
ctypes.CDLL('libc.so.6').malloc_trim(0)


def rss_gb() -> float:
    return psutil.Process().memory_info().rss / 2**30


print(f'RSS before probes: {rss_gb():.1f} GB of {psutil.virtual_memory().total / 2**30:.1f} GB')


def assign_folds(groups: list[str], k: int = FOLDS, seed: int = SEED) -> dict[str, int]:
    order = np.random.default_rng(seed).permutation(len(groups))
    return {groups[i]: int(pos % k) for pos, i in enumerate(order)}


def fit_score(pos: list[np.ndarray], neg: list[np.ndarray], tests: list[np.ndarray]) -> list[np.ndarray]:
    x = np.concatenate(pos + neg).astype(np.float64)
    y = np.concatenate([np.ones(sum(len(a) for a in pos)), np.zeros(sum(len(a) for a in neg))])
    scaler = StandardScaler(copy=False).fit(x)          # in place: same numbers, one float64 matrix
    model = LogisticRegression(C=constants.EDA_PROBE_C, max_iter=constants.EDA_PROBE_MAX_ITER,
                               class_weight='balanced', random_state=SEED)
    model.fit(scaler.transform(x), y)
    del x
    return [model.decision_function(scaler.transform(t.astype(np.float64))) for t in tests]


def pooled_auc(a: list[np.ndarray], b: list[np.ndarray]) -> float:
    # P(score of a class-`a` row > score of a class-`b` row).
    a_, b_ = np.concatenate(a), np.concatenate(b)
    return frame_auc(np.concatenate([a_, b_]), np.concatenate([np.ones(len(a_)), np.zeros(len(b_))]))


def t95(deltas: list[float]) -> dict:
    d = np.asarray(deltas)
    half = T95 * d.std(ddof=1) / np.sqrt(len(d))
    return {'mean': float(d.mean()), 'lo': float(d.mean() - half), 'hi': float(d.mean() + half),
            'per_fold': d.tolist()}


DADA_IDS = sorted(DADA_FEATS)
DADA_FOLD = assign_folds(DADA_IDS)                              # identical to the D2City run -> R0 reproduces
BDDA_FOLD = assign_folds(ARM_IDS['all'], seed=SEED + 1)         # `calm` inherits its folds from `all`
DOTA_FOLD = assign_folds(sorted(DOTA_PRE), seed=SEED + 2)
pos_rows = {v: DADA_FEATS[v][DADA_LABELS[v] == 1] for v in DADA_IDS}
neg_rows = {v: DADA_FEATS[v][DADA_LABELS[v] == 0] for v in DADA_IDS}
print(f'folds: DADA {len(DADA_IDS)} videos | BDD-A ' + ', '.join(f'{a} {len(i)}' for a, i in ARM_IDS.items())
      + f' clips | DoTA {len(DOTA_PRE)}')

In [ ]:
PROBE_NAMES = ['R0'] + [f'{p}_{arm}' for arm in ARMS for p in ('X', 'M')]


def arm_of(name: str) -> str:
    return 'all' if name == 'R0' else name.split('_', 1)[1]


def run_probes() -> dict:
    per_video = {p: {} for p in PROBE_NAMES}
    fold_macro = {p: [] for p in PROBE_NAMES}
    shortcut = {p: ([], []) for p in PROBE_NAMES}
    s_scores = {arm: ([], []) for arm in ARMS}
    sref_scores = ([], [])
    for f in range(FOLDS):
        tr = [v for v in DADA_IDS if DADA_FOLD[v] != f]
        te = [v for v in DADA_IDS if DADA_FOLD[v] == f]
        b_tr = {arm: [BDDA_FEATS[v] for v in ARM_IDS[arm] if BDDA_FOLD[v] != f] for arm in ARMS}
        b_te = {arm: [v for v in ARM_IDS[arm] if BDDA_FOLD[v] == f] for arm in ARMS}
        tests = [DADA_FEATS[v] for v in te] + [BDDA_FEATS[v] for v in b_te['all']]
        dada_neg_tr = [neg_rows[v] for v in tr]
        negs = {'R0': dada_neg_tr}
        for arm in ARMS:
            negs[f'X_{arm}'] = b_tr[arm]
            negs[f'M_{arm}'] = dada_neg_tr + b_tr[arm]
        for name, neg in negs.items():
            scores = fit_score([pos_rows[v] for v in tr], neg, tests)
            dada_scores = scores[:len(te)]
            bdda_scores = dict(zip(b_te['all'], scores[len(te):]))
            aucs = []
            for v, sc in zip(te, dada_scores):
                lab = DADA_LABELS[v]
                if 0 < lab.sum() < len(lab):
                    per_video[name][v] = frame_auc(sc, lab)
                    aucs.append(per_video[name][v])
            fold_macro[name].append(float(np.mean(aucs)))
            shortcut[name][0].extend(sc[DADA_LABELS[v] == 0] for v, sc in zip(te, dada_scores))
            shortcut[name][1].extend(bdda_scores[v] for v in b_te[arm_of(name)])
        # S: DADA pre-accident (1) vs BDD-A (0), per arm
        pre_tr = [dada_pre[v] for v in tr if v in dada_pre]
        pre_te = [dada_pre[v] for v in te if v in dada_pre]
        for arm in ARMS:
            sc = fit_score(pre_tr, b_tr[arm], pre_te + [BDDA_FEATS[v] for v in b_te[arm]])
            s_scores[arm][0].extend(sc[:len(pre_te)])
            s_scores[arm][1].extend(sc[len(pre_te):])
        if DOTA_PRE:
            dota_tr = [a for v, a in DOTA_PRE.items() if DOTA_FOLD[v] != f]
            dota_te = [a for v, a in DOTA_PRE.items() if DOTA_FOLD[v] == f]
            sc = fit_score(pre_tr, dota_tr, pre_te + dota_te)
            sref_scores[0].extend(sc[:len(pre_te)])
            sref_scores[1].extend(sc[len(pre_te):])
        print(f'  fold {f}: ' + '  '.join(f'{p} {fold_macro[p][-1]:.4f}' for p in PROBE_NAMES)
              + f'  | RSS {rss_gb():.1f} GB', flush=True)
        gc.collect()
    out = {p: {'auc_macro': float(np.mean(list(per_video[p].values()))), 'videos': len(per_video[p]),
               'fold_macro': fold_macro[p], 'shortcut_auc': pooled_auc(*shortcut[p])} for p in PROBE_NAMES}
    for arm in ARMS:
        out[f'delta_X_R0_{arm}'] = t95(np.subtract(fold_macro[f'X_{arm}'], fold_macro['R0']).tolist())
        out[f'delta_M_R0_{arm}'] = t95(np.subtract(fold_macro[f'M_{arm}'], fold_macro['R0']).tolist())
        out[f'S_{arm}'] = pooled_auc(*s_scores[arm])
    out['S_ref_DoTA'] = pooled_auc(*sref_scores) if sref_scores[0] else None
    return out


# Resumable: a finished run is on Drive; re-running this cell after a crash skips it.
probe_path = OUT / 'probes.json'
if probe_path.is_file():
    PROBES = json.loads(probe_path.read_text())
    print(f'loaded {probe_path.name}')
else:
    t0 = time.time()
    PROBES = run_probes()
    probe_path.write_text(json.dumps(PROBES, indent=2, default=float))
    print(f'probes done in {(time.time() - t0) / 60:.1f} min')
REPORT['probes'] = PROBES
save_report()

## 5. Verdict (plan §4) — read against the table, not against hope

In [ ]:
def verdict(arm: str) -> dict:
    p = PROBES
    r0, x, m = p['R0']['auc_macro'], p[f'X_{arm}']['auc_macro'], p[f'M_{arm}']['auc_macro']
    dx, dm = p[f'delta_X_R0_{arm}'], p[f'delta_M_R0_{arm}']
    sane = R0_SANITY[0] <= r0 <= R0_SANITY[1]
    if x < GX_FAIL or dx['mean'] < GX_DELTA_MARGINAL:
        gx = 'FAIL'
    elif x >= GX_PASS and dx['mean'] >= GX_DELTA_PASS:
        gx = 'PASS'
    else:
        gx = 'MARGINAL'
    gm = dm['mean'] >= GM_DELTA
    if not sane:
        call = 'PIPELINE BUG -- R0 outside the sanity band; do not read X/M'
    elif gx == 'FAIL':
        call = 'NO-GO -- BDD-A teaches source, not accident'
    elif not gm:
        call = 'NO-GO (additive) -- adding BDD-A costs localization; stay on T2'
    elif gx == 'MARGINAL':
        call = 'GO-with-in-video-negatives (M-style corpus only)'
    else:
        call = 'GO -- write the corpus-build plan'
    shortcut_x, shortcut_m = p[f'X_{arm}']['shortcut_auc'], p[f'M_{arm}']['shortcut_auc']
    return {'R0': r0, 'X': x, 'M': m, 'dX': dx, 'dM': dm, 'R0_sane': sane, 'G-X': gx, 'G-M': gm,
            'S': p[f'S_{arm}'], 'S_ref': p['S_ref_DoTA'], 'shortcut_R0': p['R0']['shortcut_auc'],
            'shortcut_X': shortcut_x, 'shortcut_M': shortcut_m,
            'shortcut_red_flag': max(shortcut_x, shortcut_m) > SHORTCUT_RED, 'call': call}


VERDICT = {arm: verdict(arm) for arm in ARMS}
r0_repro = abs(PROBES['R0']['auc_macro'] - R0_D2CITY) < R0_REPRO_TOL
REPORT['verdict'] = {'per_arm': VERDICT, 'gate_arm': GATE_ARM, 'call': VERDICT[GATE_ARM]['call'],
                     'R0_reproduces_d2city': r0_repro, 'G-I': REPORT['G-I']['pass'], 'G-L': REPORT['G-L']['pass']}
save_report()

fmt = lambda v: f'{v:.4f}' if isinstance(v, float) else str(v)
ci = lambda d: f"{d['mean']:+.4f} [{d['lo']:+.4f}, {d['hi']:+.4f}]"
a, c = VERDICT['all'], VERDICT['calm']
lines = ['# BDD-A negative-bag EDA -- read-out', '',
         f"G-I {'PASS' if REPORT['G-I']['pass'] else 'FAIL'} ({REPORT['G-I']['usable']} usable, "
         f"{REPORT['G-I']['calm']} calm) | G-L ({CHOSEN}) length AUC {REPORT['G-L']['length_auc_mix1']:.4f} "
         f"{'PASS' if REPORT['G-L']['pass'] else 'FAIL'} | T2 oracle mix0 {REPORT['G-L']['t2_oracle_mix0']:.4f}",
         f"R0 {PROBES['R0']['auc_macro']:.6f} -- reproduces D2City's {R0_D2CITY}: {r0_repro}", '',
         '| quantity | all | calm (gate) |', '|---|---:|---:|']
for key, label in (('R0', 'R0 auc_macro'), ('X', 'X auc_macro'), ('M', 'M auc_macro'),
                   ('S', 'S (DADA-pre vs BDD-A)'), ('S_ref', 'S-ref (DADA-pre vs DoTA)'),
                   ('shortcut_R0', 'shortcut AUC, R0'), ('shortcut_X', 'shortcut AUC, X'),
                   ('shortcut_M', 'shortcut AUC, M'), ('G-X', 'G-X'), ('G-M', 'G-M')):
    lines.append(f'| {label} | {fmt(a[key])} | {fmt(c[key])} |')
lines.append(f"| Δ(X−R0) t95 | {ci(a['dX'])} | {ci(c['dX'])} |")
lines.append(f"| Δ(M−R0) t95 | {ci(a['dM'])} | {ci(c['dM'])} |")
lines += ['', f'**Gate arm:** {GATE_ARM}  ·  **Call:** {VERDICT[GATE_ARM]["call"]}']
if a['call'] != c['call']:
    lines.append(f"(`all` reads differently: {a['call']} -- the braking clips matter; say so in the write-up)")
(OUT / 'eda_bdda.md').write_text('\n'.join(lines) + '\n')
print('\n'.join(lines))
print(f'\nwritten: {OUT}/eda_bdda.json, eda_bdda.md, probes.json, inventory.csv, montage.png, autocorr_seconds.png')